# Lineage Extractor Setup
Verify Fabric Environment and configure helper functions for lineage extraction.

**Prerequisites:**
- Fabric Environment with semantic-link packages attached to this notebook
- Lakehouse attached as default lakehouse
- Contributor+ role in workspace

**Environment Required Packages:**
- `semantic-link` >= 0.7.0
- `semantic-link-labs` >= 0.9.0
- `requests` >= 2.31.0

## 1. Verify Environment & Import Libraries

In [ ]:
# Verify packages are available from environment (no installation needed)
try:
    import sempy
    import sempy.fabric as fabric
    from sempy.fabric import list_items
    from sempy_labs.tom import connect_semantic_model
    import json
    import requests
    from datetime import datetime
    from typing import List, Dict, Any
    from notebookutils import mssparkutils
    
    print("✅ All packages loaded from Fabric Environment")
    print(f"   semantic-link version: {sempy.__version__}")
    
except ImportError as e:
    print("❌ Package import failed!")
    print(f"   Error: {e}")
    print("\n⚠️  Solution: Attach a Fabric Environment with required packages:")
    print("   1. Create Environment in workspace")
    print("   2. Add packages: semantic-link, semantic-link-labs, requests")
    print("   3. Publish environment")
    print("   4. Attach to this notebook (top toolbar → Environment dropdown)")
    raise

## 2. Helper Functions
Define reusable functions for all extraction notebooks.

In [ ]:
def get_fabric_token() -> str:
    """Get Fabric API access token using notebook authentication"""
    return mssparkutils.credentials.getToken("pbi")

def save_to_lakehouse(data: Any, file_path: str) -> None:
    """Save data to lakehouse Files location"""
    full_path = f"Files/{file_path}"
    
    # Ensure directory exists
    directory = "/".join(full_path.split("/")[:-1])
    mssparkutils.fs.mkdirs(directory)
    
    # Write JSON
    json_str = json.dumps(data, indent=2)
    mssparkutils.fs.put(full_path, json_str, overwrite=True)
    print(f"✅ Saved: {file_path}")

def get_workspace_artifacts(workspace_id: str, artifact_type: str) -> List[Dict]:
    """Get list of artifacts from a workspace using sempy.fabric"""
    # Use sempy.fabric.list_items() instead of REST API
    items_df = list_items(item_type=artifact_type, workspace=workspace_id)
    return items_df.to_dict('records')

print("✅ Helper functions defined")

## 4. Configuration
Set workspace and lakehouse IDs (passed as parameters from custom item).

In [ ]:
# These will be provided by the custom item when triggered via Spark Livy
# Default values for testing
WORKSPACE_IDS = ["your-workspace-id"]  # List of workspace IDs to extract from
LAKEHOUSE_ID = "your-lakehouse-id"      # Target lakehouse for storage
ARTIFACT_TYPES = ["SemanticModel", "Report", "Notebook", "Lakehouse"]

print("✅ Configuration loaded")
print(f"Workspaces: {len(WORKSPACE_IDS)}")
print(f"Artifact types: {ARTIFACT_TYPES}")

## 5. Verify Setup

In [ ]:
# Test configuration
print("\n🔍 Verifying setup...")
print(f"✅ semantic-link version: {sempy.__version__}")
print(f"✅ Fabric token available: {len(get_fabric_token()) > 0}")
print(f"✅ Lakehouse access: {mssparkutils.fs.ls('Files')}")
print("\n✅ Setup complete - ready for extraction!")

---
# 🧪 Testing Guide

## How to Test This Notebook

### 1. **Prerequisites**
- You must run this notebook in a **Fabric workspace**
- Attach a **Lakehouse** to the notebook (default lakehouse)
- Ensure you have **Contributor** or higher role in the workspace

### 2. **Update Configuration (Cell 8)**
Replace placeholder values:
```python
WORKSPACE_IDS = ["actual-workspace-guid"]  # Get from workspace URL
LAKEHOUSE_ID = "actual-lakehouse-guid"     # Get from lakehouse properties
```

### 3. **Run Cells in Order**
- Cell 3: Install packages (~30-60 seconds)
- Cell 5: Import libraries (verify no import errors)
- Cell 6: Define helper functions
- Cell 8: Set configuration
- Cell 9: Verify setup

### 4. **Expected Output**
Cell 9 should show:
```
🔍 Verifying setup...
✅ semantic-link version: 0.x.x
✅ Fabric token available: True
✅ Lakehouse access: [list of folders]
✅ Setup complete - ready for extraction!
```

### 5. **Test Helper Functions**
Run the test cells below to validate each helper function.

## 🧪 Test 1: Token Acquisition

In [ ]:
# Test Fabric authentication
try:
    token = get_fabric_token()
    print(f"✅ Token acquired: {len(token)} characters")
    print(f"   Starts with: {token[:20]}...")
except Exception as e:
    print(f"❌ Token acquisition failed: {e}")

## 🧪 Test 2: Lakehouse Write/Read

In [ ]:
# Test lakehouse file operations
try:
    test_data = {
        "test": "data",
        "timestamp": datetime.now().isoformat()
    }
    
    test_path = "lineage/test/test_file.json"
    save_to_lakehouse(test_data, test_path)
    
    # Read back
    full_path = f"Files/{test_path}"
    content = mssparkutils.fs.head(full_path, 500)
    loaded_data = json.loads(content)
    
    print(f"✅ Write/Read successful")
    print(f"   Data matches: {loaded_data == test_data}")
    
    # Cleanup
    mssparkutils.fs.rm(full_path)
    print(f"✅ Cleanup complete")
    
except Exception as e:
    print(f"❌ Lakehouse I/O failed: {e}")

## 🧪 Test 3: Fabric API Access

In [ ]:
# Test Fabric REST API calls
# Replace with an actual workspace ID you have access to
TEST_WORKSPACE_ID = WORKSPACE_IDS[0] if WORKSPACE_IDS[0] != "your-workspace-id" else None

if TEST_WORKSPACE_ID:
    try:
        # Test getting semantic models
        models = get_workspace_artifacts(TEST_WORKSPACE_ID, "SemanticModel")
        print(f"✅ API access successful")
        print(f"   Found {len(models)} Semantic Model(s) in workspace")
        
        if len(models) > 0:
            print(f"   Example: {models[0].get('displayName', 'N/A')}")
    except Exception as e:
        print(f"❌ API access failed: {e}")
        print(f"   Check workspace ID and permissions")
else:
    print("⚠️  Update TEST_WORKSPACE_ID in cell 8 to test API access")

## 🧪 Test 4: semantic-link Functions

In [ ]:
# Test semantic-link-labs functions
# Note: Requires a valid Semantic Model ID

TEST_MODEL_ID = None  # Set to actual model ID for testing

if TEST_MODEL_ID and TEST_WORKSPACE_ID:
    try:
        print("Testing semantic-link-labs...")
        
        # Test list_measures
        measures_df = list_measures(dataset=TEST_MODEL_ID, workspace=TEST_WORKSPACE_ID)
        print(f"✅ list_measures: {len(measures_df)} measures")
        
        # Test list_columns
        columns_df = list_columns(dataset=TEST_MODEL_ID, workspace=TEST_WORKSPACE_ID)
        print(f"✅ list_columns: {len(columns_df)} columns")
        
        # Test list_relationships
        rels_df = list_relationships(dataset=TEST_MODEL_ID, workspace=TEST_WORKSPACE_ID)
        print(f"✅ list_relationships: {len(rels_df)} relationships")
        
        print("\n✅ All semantic-link functions working!")
        
    except Exception as e:
        print(f"❌ semantic-link test failed: {e}")
        print(f"   Verify model ID and permissions")
else:
    print("⚠️  Set TEST_MODEL_ID to test semantic-link functions")
    print("   Get model ID from a Semantic Model in your workspace")

---
## ✅ Setup Complete

If all tests passed, you're ready to run extraction notebooks:
- `01_extract_semantic_models.ipynb` - Extract semantic model metadata
- `02_extract_reports.ipynb` - Extract report definitions

**Next Steps:**
1. Update `WORKSPACE_IDS` in cell 8 with actual workspace GUIDs
2. Verify lakehouse is attached to notebooks
3. Run extraction notebooks with valid configuration